# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HaneefAderolu/ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Cell 0 - Setup (run first every session)
import os

if not os.path.exists('ml-internship-starter'):
    os.system('git clone https://github.com/HaneefAderolu/ml-internship-starter.git')
os.chdir('ml-internship-starter')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

# Load data
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Proxy label (same as W04 and W05 - no trend columns as features)
df['label'] = (
    (df['trend_direction'] == 'down') &
    (df['impressions_90d'] >= 500) &
    (df['content_age_days'] >= 180)
).astype(int)

df['avg_position_clean'] = df['avg_position'].replace(0, np.nan)
df['has_position']        = (df['avg_position'] > 0).astype(int)
df['has_word_count']      = df['word_count'].notna().astype(int)
df['word_count_filled']   = df['word_count'].fillna(0)

features = [
    'impressions_90d','days_with_impressions','days_with_sessions',
    'avg_position_clean','has_position','ctr','engagement_rate',
    'scroll_rate','word_count_filled','has_word_count',
    'content_age_days','days_since_last_update','search_volume',
    'competition','sessions_90d','pageviews_90d',
]

X = df[features].copy().fillna(df[features].median())
y = df['label'].copy()
groups = df['client_id'].values

# Same grouped split as W05
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Same model as W05
rf = RandomForestClassifier(
    n_estimators=200, max_depth=6, min_samples_leaf=20,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

# Score all 30,000 pages
X_full       = df[features].copy().fillna(df[features].median())
rf_probs_full = rf.predict_proba(X_full)[:, 1]

print(f"Loaded {len(df):,} rows. Model ready.")
print(f"Positive label rate: {y.mean():.1%}")

Loaded 30,000 rows. Model ready.
Positive label rate: 17.9%


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*
The queue ranks every page by its Random Forest probability score (0–1).
Higher score = model believes page is more likely to match our proxy label
(declining + visible + at least 6 months old).

Six reason codes are assigned based on which combination of signals
drove the page into the top of the queue:

| Reason code            | What it means                                          | Action            |
|------------------------|-------------------------------------------------------|-------------------|
| OLD_HIGH_VOLUME        | 365+ days old, 1,000+ impressions - high-stakes stale | PRIORITISE_REVIEW |
| STALE_WEAK_POSITION    | 180+ days old, visible, position > 20                 | PRIORITISE_REVIEW |
| STALE_LOW_ENGAGEMENT   | 180+ days old, visible, engagement rate < 20%         | SCHEDULE_REVIEW   |
| HIGH_VOLUME_ACTIVE     | 2,000+ impressions, 70+ days active - healthy now     | MONITOR           |
| STALE_VISIBLE          | 180+ days old, visible, no stronger signal            | SCHEDULE_REVIEW   |
| MODERATE_RISK          | No strong signal in any dimension                     | DEPRIORITISE      |

Observed in data:
  - 9,799 pages flagged PRIORITISE_REVIEW (32.7% of scored pages)
  - 2,969 carry OLD_HIGH_VOLUME - these are the highest-stakes cases
  - Top 20 pages have a mean content age of 275 days and mean 4,653 impressions

This is decision-support, not automation. The scores say where to look first.
A human editor makes the final call.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build ranked queue with reason codes and action labels
queue = df[['content_id','client_id','impressions_90d','avg_position_clean',
            'content_age_days','engagement_rate','days_with_impressions',
            'ctr','trend_direction','label']].copy()
queue['rf_score'] = rf_probs_full
queue = queue.sort_values('rf_score', ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1

def reason_code(row):
    age = row['content_age_days']
    imp = row['impressions_90d']
    pos = row['avg_position_clean']
    eng = row['engagement_rate']
    days = row['days_with_impressions']
    if age >= 365 and imp >= 1000:
        return 'OLD_HIGH_VOLUME'
    elif age >= 180 and imp >= 500 and (pd.isna(pos) or pos > 20):
        return 'STALE_WEAK_POSITION'
    elif age >= 180 and imp >= 500 and eng < 20:
        return 'STALE_LOW_ENGAGEMENT'
    elif imp >= 2000 and days >= 70:
        return 'HIGH_VOLUME_ACTIVE'
    elif age >= 180 and imp >= 500:
        return 'STALE_VISIBLE'
    else:
        return 'MODERATE_RISK'

def action_label(score):
    if score >= 0.6:   return 'PRIORITISE_REVIEW'
    elif score >= 0.4: return 'SCHEDULE_REVIEW'
    elif score >= 0.2: return 'MONITOR'
    else:              return 'DEPRIORITISE'

queue['reason_code'] = queue.apply(reason_code, axis=1)
queue['action']      = queue['rf_score'].apply(action_label)

print("Action distribution:")
print(queue['action'].value_counts())
print("\nReason code distribution:")
print(queue['reason_code'].value_counts())
print("\nTop 20 pages:")
print(queue[['rank','impressions_90d','avg_position_clean','content_age_days',
             'rf_score','reason_code','action']].head(20).to_string())


Action distribution:
action
DEPRIORITISE         19933
PRIORITISE_REVIEW     9799
SCHEDULE_REVIEW        135
MONITOR                133
Name: count, dtype: int64

Reason code distribution:
reason_code
MODERATE_RISK           16167
STALE_LOW_ENGAGEMENT     4322
HIGH_VOLUME_ACTIVE       3949
OLD_HIGH_VOLUME          2969
STALE_WEAK_POSITION      2508
STALE_VISIBLE              85
Name: count, dtype: int64

Top 20 pages:
    rank  impressions_90d  avg_position_clean  content_age_days  rf_score           reason_code             action
0      1              840                25.4               229  0.902203   STALE_WEAK_POSITION  PRIORITISE_REVIEW
1      2             2470                 6.1               287  0.901806  STALE_LOW_ENGAGEMENT  PRIORITISE_REVIEW
2      3             2903                15.1               286  0.901408  STALE_LOW_ENGAGEMENT  PRIORITISE_REVIEW
3      4             1131                33.1               287  0.901021   STALE_WEAK_POSITION  PRIORITISE_REVIEW
4  

## 2. Intended use and limits

*Who uses this, for what - and where it stops being valid.*
INTENDED USE
  Who: a content strategist or editor at a digital marketing agency
  What: opens the ranked queue once a week, reviews the top 20–50 pages,
        decides which ones to send to writers for refresh or restructure
  When: after each monthly data export - the queue is a snapshot, not live

  The queue is a priority filter, not a decision-maker.
  It answers: "out of 30,000 pages, which 50 are worth a human looking at this week?"
  It does not answer: "will refreshing this page increase traffic?"

WHERE IT STOPS BEING VALID
  1. New clients (< 90 days of data): content_age_days and impressions_90d are
     unreliable for very new pages - the model will underrank them. Apply a separate
     "new content watch" process for pages under 90 days old.

  2. Seasonal content: a page about "Christmas gift guides" will naturally decline
     in January. The model sees decline and flags it. A human must recognise this
     and override - seasonal decay is not the same as content quality decay.

  3. Intentionally thin content (e.g. doorway pages, redirect targets): these pages
     may score high because they are old and declining, but the correct action is
     deletion, not refresh. The model has no concept of "should this page exist?"

  4. Cross-client generalisation: the model was trained on 24 clients and tested on 8.
     For a new client added after training, the model's scores should be treated as
     directional only until a re-train includes that client's data.

  5. Causation: "pages the model flags are declining" is an observed pattern.
     "Refreshing these pages will stop the decline" is a causal claim this data
     cannot support - no intervention data exists.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the limits with numbers
print("=== Limit evidence ===")

# Limit 1: new pages
new_pages = df[df['content_age_days'] < 90]
print(f"Pages under 90 days old: {len(new_pages):,} ({len(new_pages)/len(df):.1%}) — excluded from label by design")

# Limit 2: seasonal - pages with very high trend variance
print(f"\ntrend_pct range: {df['trend_pct'].min():.0f}% to {df['trend_pct'].max():.0f}%")
print(f"Pages with trend_pct > 500% (likely seasonal spikes): {(df['trend_pct'] > 500).sum():,}")

# Limit 3: cross-client - test clients vs train clients
test_clients  = df['client_id'].iloc[test_idx].unique()
train_clients = df['client_id'].iloc[train_idx].unique()
print(f"\nTrain clients: {len(train_clients)}, Test clients: {len(test_clients)}")
print("Model has never seen test clients during training - scores are generalised, not personalised")

# Limit 4: score distribution
print(f"\nScore distribution:")
print(queue['rf_score'].describe().round(3).to_string())


=== Limit evidence ===
Pages under 90 days old: 0 (0.0%) — excluded from label by design

trend_pct range: -100% to 44900%
Pages with trend_pct > 500% (likely seasonal spikes): 180

Train clients: 24, Test clients: 8
Model has never seen test clients during training - scores are generalised, not personalised

Score distribution:
count    30000.000
mean         0.290
std          0.357
min          0.000
25%          0.016
50%          0.064
75%          0.763
max          0.902


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*
WHAT A HUMAN MUST CHECK BEFORE ACTING on any PRIORITISE_REVIEW page:

  1. Is the decline seasonal?
     Check the page's topic. If it is tied to a specific time of year (holidays,
     tax season, sports events), flag it as SEASONAL and skip this cycle.

  2. Is the page still strategically relevant?
     The model does not know business strategy. A page about a discontinued product
     or a topic the client no longer covers should be archived, not refreshed.

  3. Is the traffic loss explained by a known event?
     Site migrations, algorithm updates, domain changes, and competitor launches
     can all cause declines that no content refresh will fix. Check with the client
     before ordering a rewrite.

  4. Does the page have a meaningful audience?
     A page with 500 impressions and declining trend might simply serve a very
     small niche. Refreshing it may not be worth the editorial cost.

THE NO-GO LIST - things that must never be automated:

  ✗ Auto-publishing content based on model scores
    Reason: the model flags pages worth reviewing. It cannot write or approve content.

  ✗ Deleting pages flagged as DEPRIORITISE
    Reason: low model score means "low risk signal right now" - not "this page is worthless".

  ✗ Using scores as performance KPIs for content writers
    Reason: the proxy label is a simplified construct. Writers should not be rewarded
    or penalised based on a rule-derived signal with known limitations.

  ✗ Running the model on a client's data without their knowledge
    Reason: this tool works on client content. Data governance and consent matter.

  ✗ Treating Precision@K as a guarantee
    Reason: P@20 = 0.65 means 13 of the top 20 pages match our proxy label.
    7 do not. Every single flagged page needs a human eye before action is taken.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quantify the no-go cases
print("=== Human review burden ===")
prioritise = queue[queue['action'] == 'PRIORITISE_REVIEW']
print(f"PRIORITISE_REVIEW pages: {len(prioritise):,}")
print(f"At P@20 = 0.65 → expect 13 real, 7 false positives in every top-20 batch")
print(f"At P@50 = 0.54 → expect 27 real, 23 false positives in every top-50 batch")
print()

# Show why auto-delete would be catastrophic
deprioritise = queue[queue['action'] == 'DEPRIORITISE']
high_imp_deprioritised = deprioritise[deprioritise['impressions_90d'] >= 2000]
print(f"DEPRIORITISE pages with >= 2,000 impressions: {len(high_imp_deprioritised):,}")
print("These are NOT worthless - they just lack the age+decline signal the model looks for.")
print("Auto-deleting them would be a serious mistake.")


=== Human review burden ===
PRIORITISE_REVIEW pages: 9,799
At P@20 = 0.65 → expect 13 real, 7 false positives in every top-20 batch
At P@50 = 0.54 → expect 27 real, 23 false positives in every top-50 batch

DEPRIORITISE pages with >= 2,000 impressions: 4,115
These are NOT worthless - they just lack the age+decline signal the model looks for.
Auto-deleting them would be a serious mistake.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*
The model is a snapshot trained on trailing-90-day data from 32 clients.
It will go stale. These are the signals that should trigger a review or retrain:

MONITORING CHECKS (run monthly):
  1. Score drift: if the mean rf_score of the top 100 pages drops below 0.5,
     the model may be misranking new patterns it has not seen.

  2. Precision@20 check: pull a sample of top-20 pages, have an editor review them,
     and record how many they agree are worth refreshing. If agreement drops below 50%,
     the model needs retraining.

  3. Reason code drift: if MODERATE_RISK starts dominating the top 50 (it should not),
     the score thresholds need recalibration.

  4. New client onboarding: any client with fewer than 90 days of data should be
     excluded from scoring until they have enough history.

RETRAIN TRIGGERS:
  ✗ A major Google algorithm update (ranking patterns shift - old training data misleads)
  ✗ A client portfolio change > 20% (new clients or many departing clients)
  ✗ The proxy label base rate changes by more than 5 percentage points
    (currently 17.9% of pages are labelled positive - if this jumps to 25% or drops to 12%,
     (the class balance the model was trained on is no longer representative)
  ✗ Six months have passed since last training regardless of the above

WHAT DOES NOT REQUIRE RETRAINING:
  ✓ Individual pages changing status week to week (expected - scores are a snapshot)
  ✓ One or two false positives in a weekly review batch (expected at P@20 = 0.65)
  ✓ A client adding new content (new pages score low by default - no action needed)

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show current baseline numbers that monitoring should track
print("=== Monitoring baseline (current run) ===")
print(f"Base rate (label=1):              {y.mean():.3f}  ({y.mean():.1%})")
print(f"Mean rf_score top 100:            {queue.head(100)['rf_score'].mean():.3f}")
print(f"Mean rf_score top 20:             {queue.head(20)['rf_score'].mean():.3f}")
print(f"MODERATE_RISK in top 50:          {(queue.head(50)['reason_code']=='MODERATE_RISK').sum()} pages")
print(f"Clients in training set:          {len(train_clients)}")
print(f"Clients in test set:              {len(test_clients)}")
print()

# Trigger thresholds
thresholds = {
    "retrain_if_base_rate_above": 0.22,
    "retrain_if_base_rate_below": 0.13,
    "retrain_if_top100_mean_score_below": 0.50,
    "review_if_editor_agreement_below": 0.50,
    "months_until_mandatory_retrain": 6,
}
print("Monitoring thresholds:")
for k, v in thresholds.items():
    print(f"  {k}: {v}")


=== Monitoring baseline (current run) ===
Base rate (label=1):              0.179  (17.9%)
Mean rf_score top 100:            0.895
Mean rf_score top 20:             0.899
MODERATE_RISK in top 50:          0 pages
Clients in training set:          24
Clients in test set:              8

Monitoring thresholds:
  retrain_if_base_rate_above: 0.22
  retrain_if_base_rate_below: 0.13
  retrain_if_top100_mean_score_below: 0.5
  review_if_editor_agreement_below: 0.5
  months_until_mandatory_retrain: 6


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*
Three exports for the paper:
  1. work/outputs/w07_action_queue.csv   - full ranked queue (30,000 rows), not committed to git
  2. work/figures/fig1_score_distribution.png - score distribution by action label
  3. work/figures/fig2_feature_importance.png - top 10 RF feature importances
  4. work/figures/fig3_reason_codes.png       - page count by reason code
  5. work/outputs/w07_playbook_metrics.json   - receipt of all key numbers (committed)

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# 1 - Ranked queue CSV (not committed - regenerated by this notebook)
queue.to_csv('work/outputs/w07_action_queue.csv', index=False)
print(f"✓ work/outputs/w07_action_queue.csv  ({len(queue):,} rows)")

# 2 - Score distribution figure
fig, ax = plt.subplots(figsize=(8, 4))
colors = {'PRIORITISE_REVIEW':'#d62728','SCHEDULE_REVIEW':'#ff7f0e',
          'MONITOR':'#2ca02c','DEPRIORITISE':'#aec7e8'}
for action, color in colors.items():
    subset = queue[queue['action']==action]['rf_score']
    ax.hist(subset, bins=30, alpha=0.65, label=action, color=color)
ax.set_xlabel('RF Score')
ax.set_ylabel('Number of pages')
ax.set_title('Score distribution by action label')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('work/figures/fig1_score_distribution.png', dpi=150)
plt.close()
print("✓ work/figures/fig1_score_distribution.png")

# 3 - Feature importance figure
fi = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(8, 5))
fi[::-1].plot(kind='barh', ax=ax, color='#1f77b4')
ax.set_xlabel('Gini importance')
ax.set_title('Top 10 feature importances - Random Forest')
plt.tight_layout()
plt.savefig('work/figures/fig2_feature_importance.png', dpi=150)
plt.close()
print("✓ work/figures/fig2_feature_importance.png")

# 4 - Reason code breakdown
fig, ax = plt.subplots(figsize=(8, 4))
queue['reason_code'].value_counts().plot(kind='bar', ax=ax, color='#ff7f0e', edgecolor='white')
ax.set_xlabel('Reason code')
ax.set_ylabel('Number of pages')
ax.set_title('Pages by reason code')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('work/figures/fig3_reason_codes.png', dpi=150)
plt.close()
print("✓ work/figures/fig3_reason_codes.png")

# 5 - Metrics receipt (committed to git)
def precision_at_k(sorted_df, k, col='label'):
    return sorted_df.head(k)[col].mean()

test_ids   = df['content_id'].iloc[test_idx].values
test_queue = queue[queue['content_id'].isin(test_ids)].sort_values('rf_score', ascending=False)

metrics = {
    "total_pages_ranked":        int(len(queue)),
    "action_distribution":       queue['action'].value_counts().to_dict(),
    "reason_code_distribution":  queue['reason_code'].value_counts().to_dict(),
    "precision_at_20":           round(float(precision_at_k(test_queue, 20)),  3),
    "precision_at_50":           round(float(precision_at_k(test_queue, 50)),  3),
    "precision_at_100":          round(float(precision_at_k(test_queue, 100)), 3),
    "top20_mean_rf_score":       round(float(queue.head(20)['rf_score'].mean()),           3),
    "top20_mean_age_days":       round(float(queue.head(20)['content_age_days'].mean()),   0),
    "top20_mean_impressions":    round(float(queue.head(20)['impressions_90d'].mean()),    0),
    "label_base_rate":           round(float(y.mean()), 3),
    "retrain_trigger_base_rate_upper": 0.22,
    "retrain_trigger_base_rate_lower": 0.13,
}
with open('work/outputs/w07_playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("✓ work/outputs/w07_playbook_metrics.json")
print()
print(json.dumps(metrics, indent=2))


✓ work/outputs/w07_action_queue.csv  (30,000 rows)
✓ work/figures/fig1_score_distribution.png
✓ work/figures/fig2_feature_importance.png
✓ work/figures/fig3_reason_codes.png
✓ work/outputs/w07_playbook_metrics.json

{
  "total_pages_ranked": 30000,
  "action_distribution": {
    "DEPRIORITISE": 19933,
    "PRIORITISE_REVIEW": 9799,
    "SCHEDULE_REVIEW": 135,
    "MONITOR": 133
  },
  "reason_code_distribution": {
    "MODERATE_RISK": 16167,
    "STALE_LOW_ENGAGEMENT": 4322,
    "HIGH_VOLUME_ACTIVE": 3949,
    "OLD_HIGH_VOLUME": 2969,
    "STALE_WEAK_POSITION": 2508,
    "STALE_VISIBLE": 85
  },
  "precision_at_20": 0.65,
  "precision_at_50": 0.54,
  "precision_at_100": 0.53,
  "top20_mean_rf_score": 0.899,
  "top20_mean_age_days": 275.0,
  "top20_mean_impressions": 4653.0,
  "label_base_rate": 0.179,
  "retrain_trigger_base_rate_upper": 0.22,
  "retrain_trigger_base_rate_lower": 0.13
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.